# ML Assignment 2 — Online Shoppers Purchase Prediction (training notebook)

**Name:** Tanvi Rajesh  
**BITS ID:** 2026AJ05031

This notebook trains the 5 classifiers and produces everything the Streamlit app needs: `model/*.joblib`, `model/metrics.csv`, `model/metrics.md`, `test_data.csv` and a version-pinned `requirements.txt`.

We load the project, look at the data, run `train_models.py`, check the saved models and pack the outputs into `training_outputs.zip`.

## 1. Getting the project into the notebook
On Colab we upload the zipped `online-shoppers-classifier` folder and unzip it. On the BITS Lab Jupyter we are already inside the project folder, so nothing is uploaded there.

In [ ]:
import os, zipfile, glob

try:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content")
    PROJECT = os.path.dirname(glob.glob("/content/**/train_models.py", recursive=True)[0])
except ImportError:
    # not on Colab (e.g. BITS Virtual Lab Jupyter), so we are already in the project folder
    PROJECT = os.getcwd()

os.chdir(PROJECT)
print("working in:", PROJECT)
print(sorted(os.listdir()))

## 2. Environment
We print the Python and library versions used for this run. The saved models only load reliably with the same versions, so these are what we pin in `requirements.txt` later, and what Streamlit Cloud has to match.

In [ ]:
# versions used for this run
import sys, sklearn, numpy, pandas, joblib, matplotlib, seaborn
print("Python      ", sys.version.split()[0])
for m in (sklearn, numpy, pandas, joblib, matplotlib, seaborn):
    print(f"{m.__name__:12s}", m.__version__)

## 3. Quick look at the data
We check the size, the duplicate rows, the missing values and how many sessions end in a purchase.

In [ ]:
import pandas as pd
df = pd.read_csv("data/online_shoppers_intention.csv")
print("shape:", df.shape, "| duplicates:", df.duplicated().sum(), "| missing values:", df.isnull().sum().sum())
print("\nRevenue split:\n", df["Revenue"].value_counts(normalize=True).round(3))
df.head()

What we see: 12,330 rows and 18 columns (17 features + `Revenue`), 125 duplicate rows and no missing values. Only about 15.5% of the sessions end in a purchase, so the classes are imbalanced and accuracy alone would be misleading. That is why we also look at AUC, F1 and MCC.

In [ ]:
df.describe().T.round(2)

The count and duration columns are heavily right-skewed. For example `ProductRelated_Duration` has a mean of about 1,195 seconds but a maximum of about 64,000. So in `train_models.py` we apply `log1p` to these columns before scaling them.

## 4. Train all 5 models
`train_models.py` drops the duplicates, makes a stratified 80/20 split and builds the preprocessing. It then trains the five models. Decision Tree, KNN and Naive Bayes are tuned with 5-fold cross-validation on F1. The SVM with probability outputs is the slowest part.

In [ ]:
!python train_models.py

## 5. Results
Test-set metrics of every model, side by side.

In [ ]:
import matplotlib.pyplot as plt
results = pd.read_csv("model/metrics.csv").set_index("ML Model Name")
display(results)
results.plot.bar(figsize=(11, 4.5), rot=0, title="Test-set metrics by model").legend(loc="lower right", ncol=6)
plt.ylim(0, 1); plt.tight_layout(); plt.show()

What we see:
- Logistic Regression has the best AUC, F1 and MCC.
- KNN has the highest accuracy and precision but the lowest recall, so it misses almost half of the real purchases.
- Decision Tree catches the most buyers (highest recall) but has the lowest precision.
- Naive Bayes is the weakest. With the one-hot columns it only reached an accuracy of 0.278, so we trained it on the numeric columns only (see the commit history).
- SVM is very close to Logistic Regression but slower to train.

The same table in markdown form. We copied this into the README.

In [ ]:
print(open("model/metrics.md").read())

## 6. Checking the saved models
Here we load each saved `.joblib` model and score `test_data.csv` with it, the same way the app does and without retraining. The numbers match the results table above, so the saved models are fine.

In [ ]:
# load every saved model and score the test split, like the app does
import joblib
from shopper_utils import MODEL_FILES, TARGET, clean_features, compute_metrics, to_binary

test = pd.read_csv("test_data.csv")
X_test, y_test = clean_features(test), to_binary(test[TARGET])
check = {}
for name, fname in MODEL_FILES.items():
    model = joblib.load(f"model/{fname}")
    check[name] = compute_metrics(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1])
pd.DataFrame(check).T.round(4)

## 7. Pinning the requirements
We write `requirements.txt` with the exact versions from this run, so the saved models load the same way on Streamlit Cloud.

In [ ]:
reqs = [f"scikit-learn=={sklearn.__version__}", f"numpy=={numpy.__version__}", f"pandas=={pandas.__version__}",
        f"joblib=={joblib.__version__}", f"matplotlib=={matplotlib.__version__}", f"seaborn=={seaborn.__version__}",
        "streamlit"]
open("requirements.txt", "w").write("\n".join(reqs) + "\n")
print(open("requirements.txt").read())
print("Use Python", sys.version.split()[0][:4], "on Streamlit Cloud")

## 8. Packaging the outputs
We zip the models, the test data and the requirements into `training_outputs.zip`, so they can be copied into the project folder.

In [ ]:
!zip -qr training_outputs.zip model test_data.csv requirements.txt
try:
    files.download("training_outputs.zip")
except NameError:
    print("saved training_outputs.zip in", PROJECT)